In [216]:
# Loading the data
import json
import pandas as pd
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from sklearn.covariance import MinCovDet
import numpy as np

# Define the project root (one folder up from /workshop)
project_root = Path("..").resolve()

# Add the project root to sys.path so Jupyter can find your 'src' module
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Load data/valencia_cf_1/matches.json
matches_path = project_root / "tests" / "fixtures" / "testing_data" / "data" / "valencia_cf_1" / "matches.json"
raw_matches = json.load(open(matches_path))
print(f"Loaded {len(raw_matches)} matches from {matches_path}")

Loaded 100 matches from C:\Users\kazik\projects\Gaffers-Clipboard\tests\fixtures\testing_data\data\valencia_cf_1\matches.json


In [217]:
normalized_df = pd.json_normalize(
    raw_matches,
    record_path="player_performances",
    meta=[
        "id", ["data", "half_length"], 
        ["data", "home_team_name"], 
        ["data", "away_team_name"], 
        ["data", "home_stats", "possession"], 
        ["data", "away_stats", "possession"],
        ["data", "home_stats", "xg"],
        ["data", "away_stats", "xg"],
        ["data", "home_score"],
        ["data", "away_score"],
    ]
)

normalized_df["team_possession"] = float("nan")
normalized_df["team_xg"] = float("nan")
normalized_df["opponent_xg"] = float("nan")
normalized_df["opponent_goals"] = float("nan")

for idx, row in normalized_df.iterrows():
    if row["data.home_team_name"] == "Valencia CF":
        normalized_df.at[idx, "team_possession"] = row["data.home_stats.possession"]
        normalized_df.at[idx, "team_xg"] = row["data.home_stats.xg"]
        normalized_df.at[idx, "opponent_xg"] = row["data.away_stats.xg"]
        normalized_df.at[idx, "opponent_goals"] = row["data.away_score"]
    else:
        normalized_df.at[idx, "team_possession"] = row["data.away_stats.possession"]
        normalized_df.at[idx, "team_xg"] = row["data.away_stats.xg"]
        normalized_df.at[idx, "opponent_xg"] = row["data.home_stats.xg"]
        normalized_df.at[idx, "opponent_goals"] = row["data.home_score"]

normalized_df = normalized_df.drop(columns=[
    "data.home_team_name", "data.away_team_name", "data.home_stats.possession", "data.away_stats.possession", "data.home_stats.xg", "data.away_stats.xg", "data.home_score", "data.away_score"])

normalized_df = normalized_df.rename(columns={
    "data.half_length": "half_length",
    "id": "match_id",
})

normalized_df = normalized_df[normalized_df["performance_type"] != "GK"]

normalized_df = normalized_df.dropna(axis=1, how="all")

pos_df = normalized_df[normalized_df["positions_played"].apply(lambda x: x == ["CB"])]

pos_df = pos_df.dropna(axis=1, how="all")

pos_df = pos_df.drop(columns=["performance_type", "positions_played"])

cols = pos_df.columns.tolist()
cols.insert(0, cols.pop(cols.index("match_id")))
pos_df = pos_df[cols]

print(f"Number of rows: {pos_df.shape[0]}")
print(f"Number of columns: {pos_df.shape[1]}")
print("Columns:")
for col in cols:
    print(f" - {col}")

# Output the max and min of every column, to get a sense of the range of values
for col in pos_df.columns:
    if col in ["match_id", "player_id"]:
        continue
    if pos_df[col].dtype in ["int64", "float64"]:
        print(f"{col}: min={pos_df[col].min()}, max={pos_df[col].max()}")

Number of rows: 273
Number of columns: 24
Columns:
 - match_id
 - player_id
 - goals
 - assists
 - shots
 - shot_accuracy
 - passes
 - pass_accuracy
 - dribbles
 - dribble_success_rate
 - tackles
 - tackle_success_rate
 - offsides
 - fouls_committed
 - possession_won
 - possession_lost
 - minutes_played
 - distance_covered
 - distance_sprinted
 - half_length
 - team_possession
 - team_xg
 - opponent_xg
 - opponent_goals
goals: min=0.0, max=1.0
assists: min=0.0, max=1.0
shots: min=0.0, max=4.0
shot_accuracy: min=0.0, max=100.0
passes: min=0.0, max=64.0
pass_accuracy: min=0.0, max=100.0
dribbles: min=0.0, max=32.0
dribble_success_rate: min=0.0, max=100.0
tackles: min=0.0, max=18.0
tackle_success_rate: min=0.0, max=100.0
offsides: min=0.0, max=0.0
fouls_committed: min=0.0, max=2.0
possession_won: min=0.0, max=13.0
possession_lost: min=0.0, max=8.0
minutes_played: min=1.0, max=121.0
distance_covered: min=0.0, max=15.0
distance_sprinted: min=0.0, max=7.4
team_possession: min=39.0, max=71.0


In [218]:
# 1. Select every column EXCEPT your identifiers
cols_to_convert = pos_df.columns.drop(['match_id', 'player_id'])

# 2. Force them all to numeric in one swoop
pos_df[cols_to_convert] = pos_df[cols_to_convert].apply(pd.to_numeric, errors='coerce')

# Optional: If any weird JSON text like "N/A" got turned into a NaN, 
# you can safely fill them with 0s to keep the math from breaking later.
pos_df[cols_to_convert] = pos_df[cols_to_convert].fillna(0)

In [219]:
# Step 1: Removing players with less than 10 minutes played
pos_df = pos_df[pos_df["minutes_played"] >= 10]

In [220]:
# Step 1: Volume masking
# For each percentage stat, their associated volume attempted stat must reach a certain threshold for the percentage 
# stat to be considered valid. For example, if a player attempted 0 passes, their pass accuracy should not be considered valid,
# and should be set to the mean.
volume_perc_pairs = [
    ("passes", "pass_accuracy"),
    ("shots", "shot_accuracy"),
    ("dribbles", "dribble_success_rate"),
    ("tackles", "tackle_success_rate")
]
volume_masks = {
    "passes": 5,
    "shots": 2,
    "dribbles": 3,
    "tackles": 2
}

for volume_col, perc_col in volume_perc_pairs:
    # apply the mask
    valid = pos_df[pos_df[volume_col] >= volume_masks[volume_col]]
    mean_value = valid[perc_col].mean()
    pos_df[perc_col] = pos_df.apply(
        lambda r: r[perc_col] if r[volume_col] >= volume_masks[volume_col] else mean_value,
        axis=1
    )

In [221]:
# ---------------------------------------------------------
# Step 1: Standardize Absolute Volume (The Half-Length Fix)
# ---------------------------------------------------------
cum_columns = ["goals", "assists", "shots", "passes", "dribbles", "tackles", 
               "possession_won", "possession_lost", "fouls_committed", "offsides", 
               "distance_covered", "distance_sprinted"]

h_base = 10.0

for col in cum_columns:
    # We overwrite the raw stat to represent a standard 10-minute half game.
    # This ensures 5-min half games don't corrupt our dataset averages later.
    pos_df[col] = pos_df[col] * (h_base / pos_df["half_length"])


# ---------------------------------------------------------
# Step 2: Set Bayesian Smoothing Parameters
# ---------------------------------------------------------
dummy_minutes = 30.0  # Shrinkage anchor (requires ~a half of football to break away from mean)


# ---------------------------------------------------------
# Step 3: Apply Smoothed Per-90 Scaling
# ---------------------------------------------------------
# A. Rare Stats (Assume 0 for dummy minutes)
rare_cols = ["goals", "assists", "shots"]
for col in rare_cols:
    pos_df[f"{col}_p90"] = (pos_df[col] / (pos_df["minutes_played"] + dummy_minutes)) * 90.0


# B. Volume Stats (Assume dataset average for dummy minutes)
volume_cols = ["passes", "dribbles", "tackles", "possession_won", "possession_lost", 
               "fouls_committed", "offsides", "distance_covered", "distance_sprinted"]

for col in volume_cols:
    # 1. Calculate the true, half-length-adjusted dataset average Per 90
    league_avg_p90 = (pos_df[col].sum() / pos_df["minutes_played"].sum()) * 90.0
    
    # 2. Calculate the dummy stat allocation for 45 minutes
    dummy_stat = league_avg_p90 * (dummy_minutes / 90.0)
    
    # 3. Apply the final smoothed formula
    pos_df[f"{col}_p90"] = ((pos_df[col] + dummy_stat) / (pos_df["minutes_played"] + dummy_minutes)) * 90.0

In [222]:
pos_df["xT_bonus_p90"] = 0.25 * (pos_df["distance_sprinted_p90"] / pos_df["distance_covered_p90"]) * np.log(pos_df["pass_accuracy"] * pos_df["passes_p90"] + 1)

In [223]:
# Match supremacy scalar
gamma = 0.2
delta_xg = (pos_df['team_xg'] + 1) / (pos_df['opponent_xg'] + 1)
match_supremacy_scalar = gamma * np.log(delta_xg)

In [ ]:
# Now we import weights from root/workshop/position_weights.json
# Which is in the format {"CB": {"goals_p90": 0.3, ...}, "CM": {...}, "ST": {...}}
col_names = ["goals_p90", "assists_p90", "shots_p90", "shot_accuracy", "passes_p90", "pass_accuracy",
              "dribbles_p90", "dribble_success_rate", "tackles_p90", "tackle_success_rate", "offsides_p90",
              "fouls_committed_p90", "possession_won_p90", "possession_lost_p90", "distance_covered_p90", "distance_sprinted_p90", "xT_bonus_p90"]
position_weights_path = project_root / "workshop" / "position_weights.json"
with open(position_weights_path, "r") as f:
    position_weights = json.load(f)
weights_dict = position_weights["CB"]
final_weights = np.array([weights_dict[col] for col in col_names])

In [ ]:
# Now we import the means and stds from root/workshop/position_means_stds.json
# Which is in the format {"CB": {"goals_p90": {"mean": 0.1, "std": 0.2}, ...}, "CM": {...}, "ST": {...}}
position_means_stds_path = project_root / "workshop" / "position_means_stds.json"
with open(position_means_stds_path, "r") as f:
    position_means_stds = json.load(f)
means_stds_dict = position_means_stds["CB"]

In [226]:
# Compute final z-scores
negative_stats = ["fouls_committed_p90", "possession_lost_p90", "offsides_p90"]
for col in col_names:
    mean = means_stds_dict[col]["mean"]
    std = means_stds_dict[col]["std"]
    if std == 0:
        pos_df[f"{col}_z"] = 0
    elif col in negative_stats:
        pos_df[f"{col}_z"] = (mean - pos_df[col]) / std
    else:
        pos_df[f"{col}_z"] = (pos_df[col] - mean) / std

In [227]:
# 1. The "Do No Harm" Attacking Floor 
# CBs get 0 points for not attacking, but never negative points.
attacking_stats = ['goals_p90_z', 'assists_p90_z', 'shots_p90_z', 'shot_accuracy_z']
for col in attacking_stats:
    pos_df[col] = np.clip(pos_df[col], a_min=0.0, a_max=None)

# 2. The Low-Volume Volatility Floor
# Protects against 1 missed pass or 1 missed tackle creating a mathematical black hole
efficiency_stats = ['tackle_success_rate_z', 'pass_accuracy_z', 'dribble_success_rate_z']
for col in efficiency_stats:
    pos_df[col] = np.clip(pos_df[col], a_min=-0.75, a_max=None)

In [228]:
z_score_cols = [f"{col}_z" for col in col_names]
z_matrix = pos_df[z_score_cols].values
pos_df['raw_score'] = np.dot(z_matrix, final_weights)

In [229]:
# The Set-Piece Threat (Bypassing the 0.0 PCA weight)
# A flat +0.6 for every goal, +0.4 for every assist
pos_df['raw_score'] += pos_df['goals'] * 0.6
pos_df['raw_score'] += pos_df['assists'] * 0.4

# The "Brick Wall" Dominance Bonus
# If they achieve an elite volume of defensive actions (Z-score > 2.0), give a proportional boost
pos_df['raw_score'] += np.where(pos_df['tackles_p90_z'] > 2.0, (pos_df['tackles_p90_z'] - 2.0) * 0.25, 0)
pos_df['raw_score'] += np.where(pos_df['possession_won_p90_z'] > 2.0, (pos_df['possession_won_p90_z'] - 2.0) * 0.25, 0)

# The "Playmaker CB" Bonus (Optional but highly recommended)
# Rewards the John Stones profile for hitting a lot of accurate passes from the back
pos_df['raw_score'] += np.where(pos_df['passes_p90_z'] > 1.5, (pos_df['passes_p90_z'] - 1.5) * 0.30, 0)

In [230]:
# ---------------------------------------------------------
# The Team Defense Modifiers
# ---------------------------------------------------------
# ---------------------------------------------------------
# The Advanced Clean Sheet Modifiers (xG-Adjusted)
# ---------------------------------------------------------
# 1. Base Eligibility
played_enough = pos_df['minutes_played'] >= 60
clean_sheet = pos_df['opponent_goals'] == 0

# 2. The "Lockdown" Masterclass
# They kept a clean sheet AND suffocated the opponent (e.g., at or under 1.0 xG)
lockdown_mask = played_enough & clean_sheet & (pos_df['opponent_xg'] <= 1.0)

# 3. The "Goalkeeper Bailout"
# They kept a clean sheet, but the backline was heavily breached (e.g., over 2.0 xG)
bailout_mask = played_enough & clean_sheet & (pos_df['opponent_xg'] >= 2.0)

# 4. Standard Clean Sheet
# Everything in between 1.0 and 2.0 xG
standard_cs_mask = played_enough & clean_sheet & ~lockdown_mask & ~bailout_mask

# Apply the tiered rewards to the raw score:
# Lockdown: +0.5 (Massive reward for total prevention)
pos_df['raw_score'] += np.where(lockdown_mask, 0.5, 0)

# Standard: +0.35 (The normal reward)
pos_df['raw_score'] += np.where(standard_cs_mask, 0.35, 0)

# Bailed Out: +0.15 (They get a tiny bump for surviving, but the GK was the hero)
pos_df['raw_score'] += np.where(bailout_mask, 0.15, 0)

# 2. The Defensive Meltdown Penalty
# Penalty: -0.3 to raw score.
# Condition: Opponent scores 3 or more goals. 
meltdown_mask = (pos_df['opponent_goals'] >= 3)
pos_df['raw_score'] -= np.where(meltdown_mask, 0.3, 0)

pos_df['fouls_committed_p90_z'] = np.clip(pos_df['fouls_committed_p90_z'], a_min=-2.0, a_max=None)

In [231]:
k = 0.85
s_0 = np.log(2/3) / k
# Create a new column, applying the sigmoid transformation to the raw score
pos_df['raw_rating'] = 10 * (1 / (1 + np.exp(-k * (pos_df['raw_score'] - s_0))))

In [232]:
pos_df['final_rating'] = pos_df['raw_rating'] - match_supremacy_scalar

In [233]:
pos_df['final_rating'] = pos_df['final_rating'].apply(lambda x: max(0, min(10, x)))

In [250]:
random_row = pos_df.sample(n=1).iloc[0]
for col in pos_df.columns:
    print(f"{col}: {random_row[col]:.2f},")

match_id: 15.00,
player_id: 11.00,
goals: 0.00,
assists: 0.00,
shots: 0.00,
shot_accuracy: 47.17,
passes: 30.00,
pass_accuracy: 100.00,
dribbles: 23.00,
dribble_success_rate: 100.00,
tackles: 17.00,
tackle_success_rate: 41.00,
offsides: 0.00,
fouls_committed: 1.00,
possession_won: 11.00,
possession_lost: 0.00,
minutes_played: 92.00,
distance_covered: 10.70,
distance_sprinted: 3.30,
half_length: 10.00,
team_possession: 58.00,
team_xg: 2.60,
opponent_xg: 2.50,
opponent_goals: 2.00,
goals_p90: 0.00,
assists_p90: 0.00,
shots_p90: 0.00,
passes_p90: 26.63,
dribbles_p90: 19.85,
tackles_p90: 14.17,
possession_won_p90: 9.48,
possession_lost_p90: 0.32,
fouls_committed_p90: 0.83,
offsides_p90: 0.00,
distance_covered_p90: 10.33,
distance_sprinted_p90: 3.14,
xT_bonus_p90: 0.60,
goals_p90_z: 0.00,
assists_p90_z: 0.00,
shots_p90_z: 0.00,
shot_accuracy_z: 0.00,
passes_p90_z: 1.37,
pass_accuracy_z: 1.15,
dribbles_p90_z: 1.75,
dribble_success_rate_z: 0.57,
tackles_p90_z: 2.94,
tackle_success_rate_z: -0.